# Verificar GPU y recursos

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import shutil
total, used, free = shutil.disk_usage('/workspace')
print(f'Disco total: {total/1e9:.1f} GB')
print(f'Disco libre: {free/1e9:.1f} GB')


# Instalar dependencias

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'nnunetv2', 'nibabel', 'scipy',
                'pandas', 'tqdm', 'psutil', 'nvidia-ml-py', '-q'])
print('Dependencias instaladas')


# Configurar rutas

In [ ]:
import os
from pathlib import Path

WORKSPACE      = '/workspace'
NNUNET_RAW     = f'{WORKSPACE}/nnUNet_raw'
NNUNET_PREPROC = f'{WORKSPACE}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{WORKSPACE}/nnUNet_results'
DATASET_NAME   = 'Dataset507_VerSe2020'
CONFIG         = '3d_lowres'

DRIVE_BASE    = '/content/drive/MyDrive/VerSe_2020_Dataset'
DRIVE_NNUNET  = f'{DRIVE_BASE}/SwinUNETR_training/swinunetr'
DRIVE_PREPROC = f'{DRIVE_BASE}/preprocessed_verse_for_training'

for path in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS]:
    Path(path).mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
print('Rutas configuradas')


# Copiar preprocessing desde Drive

In [ ]:
import subprocess
from pathlib import Path

def rclone_copy(src, dst, desc='Copiando'):
    Path(dst).mkdir(parents=True, exist_ok=True)
    print(f'{desc}...')
    result = subprocess.run(
        ['rclone', 'copy', src, dst, '--progress', '--transfers=16'],
        capture_output=True, text=True
    )
    n = len(list(Path(dst).rglob('*')))
    print(f'  {n} archivos copiados')

rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_raw/{DATASET_NAME}',
    f'{NNUNET_RAW}/{DATASET_NAME}',
    'Copiando nnUNet_raw'
)
rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}',
    f'{NNUNET_PREPROC}/{DATASET_NAME}',
    'Copiando preprocessed'
)
print('Preprocessing listo')


# Ajustar batch size

In [ ]:
import json
from pathlib import Path

plans_path = Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans.json'
with open(plans_path) as f:
    plans = json.load(f)

plans['configurations']['3d_lowres']['batch_size'] = 2

with open(plans_path, 'w') as f:
    json.dump(plans, f, indent=2)
print(f'batch_size: {plans["configurations"]["3d_lowres"]["batch_size"]}')


# Instalar trainer SwinUNETR 250 epochs

In [ ]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = "import torch\nfrom nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerSwinUNETR import nnUNetTrainerSwinUNETR\n\nclass nnUNetTrainerSwinUNETR_250epochs(nnUNetTrainerSwinUNETR):\n    def __init__(self, plans, configuration, fold, dataset_json,\n                 unpack_dataset=True, device=torch.device('cuda')):\n        super().__init__(plans, configuration, fold, dataset_json,\n                         unpack_dataset, device)\n        self.num_epochs = 250\n"

trainer_path = trainer_dir / 'nnUNetTrainerSwinUNETR_250epochs.py'
with open(trainer_path, 'w') as f:
    f.write(trainer_code)
print(f'Trainer guardado en {trainer_path}')


# Configurar wandb

In [ ]:
import wandb
import os

WANDB_API_KEY = 'TU_WANDB_API_KEY'  # reemplaza con tu key
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY)
print('wandb configurado')


# Entrenar 5 folds

In [ ]:
import subprocess
import wandb

wandb.init(
    project='verse-spine-segmentation',
    name='swinunetr-3d-lowres',
    config={
        'model':   'SwinUNETR',
        'config':  CONFIG,
        'trainer': 'nnUNetTrainerSwinUNETR_250epochs',
        'epochs':  250,
        'folds':   5,
    }
)

for fold in range(5):
    print(f'Entrenando fold {fold}...')
    cmd = [
        'nnUNetv2_train', DATASET_NAME, CONFIG, str(fold),
        '-tr', 'nnUNetTrainerSwinUNETR_250epochs', '--npz'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'Error en fold {fold}: {result.stderr[-300:]}')
    else:
        print(f'Fold {fold} completado')

wandb.finish()
print('Entrenamiento completo')


# Guardar checkpoints en Drive

In [ ]:
import shutil
from pathlib import Path

trainer_folder = f'nnUNetTrainerSwinUNETR_250epochs__nnUNetPlans__{CONFIG}'
src_base = Path(NNUNET_RESULTS) / DATASET_NAME / trainer_folder
dst_base = Path(DRIVE_NNUNET)   / DATASET_NAME / trainer_folder
dst_base.mkdir(parents=True, exist_ok=True)

for fold in range(5):
    src = src_base / f'fold_{fold}'
    dst = dst_base / f'fold_{fold}'
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'fold_{fold} guardado')
    else:
        print(f'fold_{fold} no encontrado')

print('Checkpoints guardados en Drive')
